<a href="https://colab.research.google.com/github/ZoodiacR/CASF/blob/main/3_3_3_THEORY_Analyzing_E_commerce_data_with_SQL_and_Pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Importing the necessary libraries
import sqlite3
import pandas as pd

For the following excerecise we will be using sqlite3 to interact with SQL.

You will see different functions and methods.

The connect() function returns a connection object that we will use to interact with the SQLite database held in the file olistdb.

To execute SQL statements and fetch results from SQL queries, we will need to use a database cursor, the .cursor().

Then, we can write the code to be executed in the .execute() method. We will focus our attention on the content of these queries, as these can be used not only with sqlite3 (it can be used on an .sql file, among others).

Finally, to return the results we need to use the fetchall() method.

In [ ]:
!gdown "153roT3S8vI2p07ok0xJHn8qq8S22mxHK"

# Creating the connection object
conn = sqlite3.connect('olist.db')

# Creating the cursor
c = conn.cursor()

Downloading...
From (original): https://drive.google.com/uc?id=153roT3S8vI2p07ok0xJHn8qq8S22mxHK
From (redirected): https://drive.google.com/uc?id=153roT3S8vI2p07ok0xJHn8qq8S22mxHK&confirm=t&uuid=ffe37cfb-f915-4944-bc5c-3a6a6223ca32
To: /content/olist.db
100% 139M/139M [00:01<00:00, 91.5MB/s]


In [ ]:
cursor = c.execute(
  """
  SELECT name
  FROM sqlite_master
  WHERE type='table';
  """
)

# Then we can retreive the results using fetchall()
rows = cursor.fetchall()
rows

[('olist_customers',),
 ('olist_geolocation',),
 ('olist_order_items',),
 ('olist_order_payments',),
 ('olist_order_reviews',),
 ('olist_orders',),
 ('olist_products',),
 ('olist_sellers',),
 ('product_category_name_translation',),
 ('public_holidays',)]

First, we are going to calculate the amount of sales per state, all present in the same dataset, olist customers. Every order has the name of the state where it happened, so we will be counting them, grouping them and ordering them in descending order in the following query:

In [ ]:
# Executing the query
c.execute('''
        SELECT customer_state, COUNT (customer_state) AS Amount
        FROM olist_customers
        GROUP BY customer_state
        ORDER BY Amount DESC
        ''')

# Fetching the results and creating a Dataframe with them
display (pd.DataFrame(c.fetchall(),
                      columns=['Customer State',
                               'Amount of Customers per State']).set_index('Customer State'))

,Amount of Customers per State
Customer State,
SP,41746
RJ,12852
MG,11635
RS,5466
PR,5045
SC,3637
BA,3380
DF,2140
ES,2033


Now, we are going to calculate the average review scoring of the orders per month and year. We will use the database called olist order reviews, that contains the review creation date (the date when the review was created) and the review score (from 1 to 5). For this we will have to select the month and the year of the review dates, and calculate the average of the review score, and group by the month of each year in the following query:

In [ ]:
# Executing the query
c.execute('''
        WITH review_time AS
        (SELECT strftime('%m', review_creation_date) AS month_number, strftime('%Y', review_creation_date) AS year, review_score
        FROM olist_order_reviews)
        SELECT month_number,
        AVG(CASE WHEN year = '2016' THEN review_score END),
        AVG(CASE WHEN year = '2017' THEN review_score END),
        AVG(CASE WHEN year = '2018' THEN review_score END)
        FROM review_time
        GROUP BY month_number;
        ''')

# Fetching the results and creating a Dataframe with them
display (pd.DataFrame(c.fetchall(),
                    columns=['Month',
                             '2016 Review Avg Score',
                             '2017 Review Avg Score',
                             '2018 Review Avg Score']))

,Month,2016 Review Avg Score,2017 Review Avg Score,2018 Review Avg Score
0,01,NaN,4.338912,4.063603
1,02,NaN,4.280962,4.014260
2,03,NaN,4.033051,3.727413
3,04,NaN,4.036983,3.919857
4,05,NaN,4.100539,4.193081
5,06,NaN,4.127616,4.197171
6,07,NaN,4.183271,4.288250
7,08,NaN,4.224961,4.209747
8,09,NaN,4.182424,NaN
9,10,4.055866,4.182188,NaN


In [ ]:
# Executing the query
c.execute('''
        WITH review_time AS
        (SELECT
	        strftime('%m', review_creation_date) AS month_number,
	        strftime('%Y', review_creation_date) AS year,
                review_score
        FROM olist_order_reviews)
        SELECT year, month_number,
        AVG(review_score)
        FROM review_time
        GROUP BY year, month_number
        ORDER BY year DESC, month_number DESC;
        ''')

# Fetching the results and creating a Dataframe with them
display (pd.DataFrame(c.fetchall(),
                    columns=['Year',
                             'Month',
                             'Review Avg Score']))

,Year,Month,Review Avg Score
0,2018,08,4.209747
1,2018,07,4.288250
2,2018,06,4.197171
3,2018,05,4.193081
4,2018,04,3.919857
5,2018,03,3.727413
6,2018,02,4.014260
7,2018,01,4.063603
8,2017,12,3.931596
9,2017,11,4.114919


Finally, we are going to calculate the total amount of delivered orders per city (top 15). For this we will have to join two datasets that contain all the info we want. Olist orders has the status of the orders and olist customers has the customer city. They both share a column called customer_id, that we will use to join them. We will count the number of cities, we will filter those that only have the "delivered" status, group by the customer cities and order everything by descending amount:

In [ ]:
# Executing the query
c.execute('''
        SELECT customer_city, COUNT (customer_city) AS Amount
        FROM olist_customers
        JOIN olist_orders ON olist_customers.customer_id = olist_orders.customer_id
        WHERE order_status = 'delivered'
        GROUP BY customer_city
        ORDER BY Amount DESC
        LIMIT 15;
        ''')

# Fetching the results and creating a Dataframe with them
display (pd.DataFrame(c.fetchall(), columns=['Customer City',
                                           'Amount of Delivered Orders']))

,Customer City,Amount of Delivered Orders
0,sao paulo,15045
1,rio de janeiro,6601
2,belo horizonte,2697
3,brasilia,2071
4,curitiba,1489
5,campinas,1406
6,porto alegre,1342
7,salvador,1188
8,guarulhos,1144
9,sao bernardo do campo,911
